# Advanced Deep Learning AMR System Exploration

This notebook explores the performance and architecture of the multi-source deep learning model trained using `src/ml/advanced_dl_system.py`. This model integrates k-mer features from NCBI, CARD, PATRIC, and ResFinder datasets and uses an advanced Residual Network with Multi-Head Attention and Focal Loss.

## Setup and Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from sklearn.metrics import classification_report, f1_score, roc_auc_score, hamming_loss
from sklearn.preprocessing import StandardScaler

# Set device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: mps


## Model Architecture (Copied from advanced_dl_system.py)

We re-define the architecture classes here for a self-contained notebook.

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, x):
        x = x.unsqueeze(1)
        batch_size = x.size(0)
        
        q = self.q_linear(x).view(batch_size, -1, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_linear(x).view(batch_size, -1, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_linear(x).view(batch_size, -1, self.n_heads, self.d_head).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)
        
        context = torch.matmul(attn, v).transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_head)
        return self.out_linear(context).squeeze(1)

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.gelu(x + self.net(x))

class AdvancedDeepAMR(nn.Module):
    """Advanced Deep Learning Model for AMR Prediction."""
    
    def __init__(self, input_dim, output_dim, hidden_dim=512, n_blocks=4):
        super().__init__()
        
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU()
        )
        
        self.attention = MultiHeadAttention(hidden_dim, n_heads=8)
        
        self.res_blocks = nn.ModuleList([
            ResidualBlock(hidden_dim) for _ in range(n_blocks)
        ])
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, output_dim)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x + self.attention(x)
        for block in self.res_blocks:
            x = block(x)
        return self.classifier(x)

## Data Loading and Preprocessing

Since the saved model relies on the unified data structure and scaler, we reload the relevant test data files and re-apply the custom loading logic from the training script.

In [3]:
def load_unified_test_data(data_root="data/processed", unified_drug_classes=None, scaler=None) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Re-loads and aligns the test data exactly as done in training."""
    data_root = Path(data_root)
    sources = ["ncbi", "card", "patric", "resfinder"]
    merged_data = {"X": [], "y": []}
    
    for source in sources:
        source_dir = data_root / source
        if not source_dir.exists(): continue
            
        for meta_path in source_dir.glob("*_metadata.json"):
            prefix = meta_path.name.replace("_metadata.json", "")
            
            with open(meta_path) as f:
                meta = json.load(f)
                
            source_classes = meta.get("class_names", []) or meta.get("drug_classes", [])
            class_map = {cls: unified_drug_classes.index(cls) for cls in source_classes if cls in unified_drug_classes}
            
            if not class_map: continue
            
            x_path = source_dir / f"{prefix}_X_test.npy"
            y_path = source_dir / f"{prefix}_y_test.npy"
            
            if not (x_path.exists() and y_path.exists()): continue
                
            X = np.load(x_path)
            y_orig = np.load(y_path)
            
            # Align Y to unified label space (same logic as in trainer)
            y_aligned = np.zeros((len(y_orig), len(unified_drug_classes)))
            
            if y_orig.ndim == 1:
                # Multiclass (one label per sample) - convert to multi-label format
                for i, label_idx in enumerate(y_orig):
                    if label_idx < len(source_classes):
                        cls = source_classes[label_idx]
                        if cls in class_map:
                            y_aligned[i, class_map[cls]] = 1
            else:
                # Already multi-label
                for old_idx, cls in enumerate(source_classes):
                    if cls in class_map:
                        new_idx = class_map[cls]
                        y_aligned[:, new_idx] = y_orig[:, old_idx]
                        
            merged_data["X"].append(X)
            merged_data["y"].append(y_aligned)
            
    X_test = np.vstack(merged_data["X"])
    y_test = np.vstack(merged_data["y"])
    
    # Apply saved scaler
    X_test_scaled = scaler.transform(X_test)
    
    print(f"Loaded {X_test_scaled.shape[0]} test samples with {X_test_scaled.shape[1]} features.")
    
    return X_test_scaled, y_test, unified_drug_classes

## Model and Data Loading

In [4]:
# Load Checkpoint
CHECKPOINT_PATH = "models/advanced_deepamr_system.pt"
try:
    # Note: weights_only=False is used to load non-PyTorch objects (StandardScaler)
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
except Exception as e:
    print(f"Failed to load checkpoint: {e}")
    # Fallback/alternative: attempt to add necessary globals if needed (for user's environment)
    from sklearn.preprocessing import StandardScaler
    torch.serialization.add_safe_globals([StandardScaler])
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)

scaler = checkpoint['scaler']
unified_classes = checkpoint['classes']
input_dim = scaler.mean_.shape[0]
output_dim = len(unified_classes)

# Instantiate Model
model = AdvancedDeepAMR(input_dim=input_dim, output_dim=output_dim)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

print(f"Model loaded with {input_dim} features and {output_dim} classes.")

# Load Data
X_test, y_test_true, classes = load_unified_test_data(
    unified_drug_classes=unified_classes, 
    scaler=scaler
)

test_tensor = torch.FloatTensor(X_test).to(DEVICE)
print(f"Test data shape: {test_tensor.shape}")

Model loaded with 500 features and 11 classes.
Loaded 864 test samples with 500 features.
Test data shape: torch.Size([864, 500])


## Inference and Metrics

In [5]:
with torch.no_grad():
    logits = model(test_tensor)
    y_prob = torch.sigmoid(logits).cpu().numpy()
    y_pred = (y_prob > 0.5).astype(int)
    
# Calculate Metrics
micro_f1 = f1_score(y_test_true, y_pred, average='micro')
macro_f1 = f1_score(y_test_true, y_pred, average='macro')
micro_auc = roc_auc_score(y_test_true, y_prob, average='micro')
macro_auc = roc_auc_score(y_test_true, y_prob, average='macro')
h_loss = hamming_loss(y_test_true, y_pred)

print("\n--- Overall Test Performance (DeepAMR System) ---")
print(f"Micro F1 (Overall Accuracy): {micro_f1:.4f}")
print(f"Macro F1 (Imbalance Handled): {macro_f1:.4f}")
print(f"Micro AUC: {micro_auc:.4f}")
print(f"Macro AUC: {macro_auc:.4f}")
print(f"Hamming Loss: {h_loss:.4f}")
print("--------------------------------------------------")


--- Overall Test Performance (DeepAMR System) ---
Micro F1 (Overall Accuracy): 0.8426
Macro F1 (Imbalance Handled): 0.6999
Micro AUC: 0.9855
Macro AUC: 0.9785
Hamming Loss: 0.0437
--------------------------------------------------


## Detailed Per-Class Analysis

A detailed classification report helps identify which drug classes are performing well (common classes) and which are still challenging (rare classes, indicated by low F1 scores and support).

In [6]:
report = classification_report(y_test_true, y_pred, target_names=classes, zero_division=0)
print("\n--- Detailed Classification Report ---")
print(report)


--- Detailed Classification Report ---
                precision    recall  f1-score   support

aminoglycoside       0.82      0.88      0.85       191
   beta-lactam       0.96      0.98      0.97       538
    fosfomycin       0.96      0.81      0.88        58
  glycopeptide       0.87      0.87      0.87        15
     macrolide       0.92      0.41      0.57       119
      phenicol       0.88      0.56      0.69        82
     quinolone       0.97      0.66      0.79        56
    rifampicin       0.00      0.00      0.00         6
   sulfonamide       0.60      0.84      0.70        81
  tetracycline       0.75      0.85      0.80       144
  trimethoprim       0.85      0.45      0.59        73

     micro avg       0.87      0.82      0.84      1363
     macro avg       0.78      0.66      0.70      1363
  weighted avg       0.88      0.82      0.83      1363
   samples avg       0.79      0.78      0.78      1363

